# Layer Normalization

it is different from the batch normalization . In batch normalization normalization depend on the batch but in layer normalization its not depend on the batch size.

Why we need Layer normalization.

1. Gradient is TOO high(exploid GD) or TOO low(vanishing GD).
2. output distribution change  of the neuron .

Both happend bec of the neuron output is uncertain. 
Due to this Unstable Training .

Layer Normalization make this training stable by controling the output of the neuron.

neuron outputs mean=0 and varience =1 . due to which stable output of the neuron 

In [87]:
import torch 
import torch.nn as nn 
torch.manual_seed(123)
batch_ex=torch.rand(2,5)
nor_layer=nn.Sequential(nn.Linear(5,6),nn.ReLU())
## input =5 , output=neuron =6 
out=nor_layer(batch_ex)
out

tensor([[0.0000, 0.0000, 0.4091, 0.6587, 0.3914, 0.0000],
        [0.0000, 0.0000, 0.1902, 0.3182, 0.6486, 0.0000]],
       grad_fn=<ReluBackward0>)

the value is +ve bec, of ReLu Act which make the -Ve into the 0

In [88]:
mean=out.mean(dim=-1,keepdim=True)
var=out.var(dim=-1, keepdim=True)
print("mean=\n",mean)
print("var\n",var)

mean=
 tensor([[0.2432],
        [0.1928]], grad_fn=<MeanBackward1>)
var
 tensor([[0.0799],
        [0.0670]], grad_fn=<VarBackward0>)


Make Normalized Layer

In [89]:
out_norm=(out-mean)/torch.sqrt(var)
print(out_norm)
mean_norm=out_norm.mean(dim=-1,keepdim=True)
var_norm=out_norm.var(dim=-1,keepdim=True)
print("mean_norm\n",mean_norm)
print("var_norm\n",var_norm)

tensor([[-0.8603, -0.8603,  0.5869,  1.4698,  0.5242, -0.8603],
        [-0.7450, -0.7450, -0.0102,  0.4844,  1.7608, -0.7450]],
       grad_fn=<DivBackward0>)
mean_norm
 tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)
var_norm
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


e-09 is the sciententic notation but we need eaxct 0 . 

In [90]:
torch.set_printoptions(sci_mode=False)
mean_norm

tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)

### Make the Class

In [91]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps=1e-5
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        self.scale = nn.Parameter(torch.ones(emb_dim))

    def forward(self,out):
        mean=out.mean(dim=-1,keepdim=True)
        var=out.var(dim=-1,keepdim=True,unbiased=False)
        out_norm=(out-mean)/torch.sqrt(var+self.eps)
        return self.scale*out_norm + self.shift 
                     ## element wise multiplication and addition



unbiased = False means

mean=sum/n and var both

when Unbiased=True then (bessels correction)

mean=sum/n-1 and var also

In [92]:
lay=LayerNorm(emb_dim=5)
out_lay=lay(batch_ex)
print(out_lay.mean(dim=-1,keepdim=True))
print(out_lay.var(dim=-1,keepdim=True,unbiased=False))
print(out_lay.var(dim=-1,keepdim=True,unbiased=True))

tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)
tensor([[0.9998],
        [0.9999]], grad_fn=<VarBackward0>)
tensor([[1.2497],
        [1.2499]], grad_fn=<VarBackward0>)
